# Notebook 2: choose a model and train it

We continue from notebook 1 (`workshop/data/prepared.npz`). The task: **from the last 24 h of inlet measurements (flow, ammonium, COD), forecast the flow for the next 2 h.**

| Step | What | Repository |
|---|---|---|
| 1 | the shapes decide the model | `forecast/models.py`, `warning/models.py` |
| 2 | baselines first | `warning/baselines.py` |
| 3 | a linear model; counting parameters | |
| 4 | an MLP, layer by layer | `rl/ddpg.py` (`_mlp`), `sacil/sacil.py` (`mlp`) |
| 5 | the training loop | `forecast/train.py`, `warning/train.py` |
| 6 | a causal convolutional network | `forecast/models.py` (`CausalBlock`) |
| 7 | overfitting, and how to stop it | |
| 8 | the test, once; save and reload | `forecast/train.py:85`, `forecast/predictor.py` |
| 9 | (extra) forecast a range, not a number: quantiles | `forecast/models.py` (`quantile_loss`) |


---
**New to notebooks? Read this first.**

- A notebook is a page made of **cells**. Grey cells hold Python code; white cells (like this one) hold text.
- Click a code cell and press **Shift + Enter**: the code runs and its result appears right under it. Then the next cell is selected.
- Run the cells **in order, from the top**. A cell can use things made by the cells above it.
- If something goes wrong: menu **Kernel → Restart Kernel and Run All Cells** starts everything again from a clean state.
- You cannot break anything. Change a number, run the cell again, and see what happens: that is how you learn.

> 🧒 **In plain words:** a **model** is a recipe with amounts you can adjust: a pinch more salt, a little less sugar. Those adjustable amounts are the **parameters** (or *weights*). **Training** is tasting and adjusting, again and again, until the dish tastes right. "Tasting" means comparing the model's guess with the true answer and measuring how far off it was: that measure is the **loss**.

**Words used in this notebook**

| word | meaning |
|---|---|
| **model** | a function with adjustable numbers inside, which turns an input into a prediction |
| **parameter / weight** | one adjustable number inside the model (a knob) |
| **baseline** | a very simple rule that needs no learning; a model must beat it to be worth anything |
| **layer** | one step of processing inside a neural network; data passes through the layers one by one |
| **neuron** | one unit of a layer: it multiplies its inputs by weights, adds them up, and passes the result on |
| **activation (ReLU)** | the rule applied after the sum; ReLU turns negative numbers into 0 |
| **loss** | one number that says how wrong the model is; training makes it smaller |
| **gradient** | for every knob, which direction makes the loss smaller, and how strongly |
| **learning rate** | how big a step we take each time we turn the knobs |
| **batch** | a handful of examples looked at together before one adjustment |
| **epoch** | one full pass through all the training examples |
| **overfitting** | memorising the training examples instead of learning the pattern |
| **MAE** | mean absolute error: the average size of the mistakes, here in m³ per day |

In [ ]:
import copy, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

torch.manual_seed(0); np.random.seed(0)
torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (11, 3.4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
DATA = Path("../data")

## 1. Look at the shapes: they decide the model


> 🧒 **In plain words:** before choosing a tool, look at the job. Here the input is a **sequence** (a row of numbers in time order) and the output is **8 numbers**. That tells us which kinds of model make sense, like knowing whether you need a spoon or a fork before you reach into the drawer.

In [ ]:
d = np.load(DATA / "prepared.npz")
X = {s: d[f"X_{s}"] for s in ("train", "val", "test")}
Y = {s: d[f"y_{s}"] for s in ("train", "val", "test")}
T = {s: d[f"t_{s}"] for s in ("train", "val", "test")}
MEAN, STD, LOOKBACK, HORIZON = d["mean"], d["std"], int(d["lookback"]), int(d["horizon"])
for s in X:
    print(f"{s:5s} X {X[s].shape}  →  y {Y[s].shape}")

**Reading the shapes:**
- input `(N, 96, 3)`: a **sequence** (96 time steps) of 3 channels → models for tables can use it flattened (96 × 3 = 288 numbers); models for sequences can use the time order (convolutions, recurrent networks).
- output `(N, 8)`: **8 numbers** → regression; the last layer is linear with 8 outputs; the loss is MSE or MAE.
- **N ≈ 1 900 training examples**: small. That limits how big the model can be (step 7).

All errors below are reported in **physical units (m³/d)**, after undoing the scaling, so they mean something to an engineer.

In [ ]:
def to_flow(z):
    """Scaled log-flow → m³/d (undo notebook 1's transform)."""
    return np.exp(np.asarray(z) * STD[0] + MEAN[0])


def mae(pred_z, split):
    """Mean absolute error per horizon (15 min … 2 h), in m³/d."""
    return np.abs(to_flow(pred_z) - to_flow(Y[split])).mean(0)


results = {}

def report(name, pred, n_params, split="val"):
    e = mae(pred, split)
    results[name] = {"params": n_params, f"{split} MAE 15 min": e[0], f"{split} MAE 2 h": e[-1]}
    print(f"{name:28s} params {n_params:>9,}   {split} MAE: 15 min {e[0]:7.0f}   2 h {e[-1]:7.0f}  m³/d")

## 2. Baselines first

Before any learning: how good are rules that need no training at all?
- **persistence**: the next 2 h equal the last value;
- **same time yesterday**: the input is exactly 24 h long, so "yesterday at the target time" is `x[h]`.


> 🧒 **In plain words:** before building a weather-forecasting machine, check how often "tomorrow will be like today" is right. If your machine can't beat that, it is useless. These no-learning guesses are the **baselines**.

In [ ]:
def persistence(x):
    return np.repeat(x[:, -1:, 0], HORIZON, axis=1)


def yesterday(x):
    return x[:, :HORIZON, 0]


for s in ("val",):
    report("persistence", persistence(X[s]), 0, s)
    report("same time yesterday", yesterday(X[s]), 0, s)

Persistence is hard to beat at 15 minutes; yesterday knows the daily shape. A model has to beat **both** to earn its place.

## 3. A linear model: the simplest learner

Flatten each window into 288 numbers and learn `ŷ = W·x + b`. Parameters: 288 × 8 weights + 8 biases.


> 🧒 **In plain words:** the simplest learner: every input gets a weight, multiply and add up. It's like a shopping bill: price × quantity for each item, then the total. Learning means finding the best "prices" (weights).

In [ ]:
flat = lambda x: x.reshape(len(x), -1)
lin = Ridge(alpha=1.0).fit(flat(X["train"]), Y["train"])
n_lin = lin.coef_.size + lin.intercept_.size
print("weights", lin.coef_.shape, "+ biases", lin.intercept_.shape, "=", n_lin, "parameters")
report("linear (ridge)", lin.predict(flat(X["val"])), n_lin)

Which inputs does the linear model use? Its weights for the 2-hour-ahead output, over the 96 input steps:

In [ ]:
w = lin.coef_[-1].reshape(LOOKBACK, 3)
plt.plot(np.arange(-LOOKBACK, 0) / 4, w, label=["Q", "S_NH", "S_S"])
plt.xlabel("hours before now"); plt.ylabel("weight"); plt.legend(); plt.title("linear model: weight of each input for the 2-h forecast")

## 4. A neural network (MLP), and what happens inside it

Same input (288 numbers), but two hidden layers with a non-linearity (ReLU). This is the kind of network the repository uses for its RL agents (`rl/ddpg.py`: 3 → 256 → 256 → 1).


> 🧒 **In plain words:** a neural network is several linear models stacked, with a small twist between them (ReLU: "if negative, make it 0"). The twist is what lets it learn curved, complicated patterns. Data goes in at the top and passes **layer by layer**, like a product on an assembly line, each station doing one small job.

In [ ]:
def mlp(hidden=(64, 64), dropout=0.0):
    layers, n_in = [nn.Flatten()], LOOKBACK * 3
    for h in hidden:
        layers += [nn.Linear(n_in, h), nn.ReLU()] + ([nn.Dropout(dropout)] if dropout else [])
        n_in = h
    layers.append(nn.Linear(n_in, HORIZON))
    return nn.Sequential(*layers)


def count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


torch.manual_seed(0)
net = mlp()
print(net)
print("\nparameters per layer:")
for name, p in net.named_parameters():
    print(f"  {name:10s} {str(tuple(p.shape)):12s} {p.numel():>7,}")
print("  total", f"{count(net):,}")

### Follow one example through the network, layer by layer

This is the forward pass from the slides, with real numbers.


> 🧒 **In plain words:** watch the **shape** change at each station of the assembly line: 96 × 3 numbers are flattened into one line of 288, squeezed to 64, then to 64 again, and finally to the 8 forecasts.

In [ ]:
x1 = torch.from_numpy(X["val"][:1])          # one example: (1, 96, 3)
h = x1
print(f"{'input':28s} shape {tuple(h.shape)}")
for layer in net:
    h = layer(h)
    vals = h.detach().numpy().ravel()
    zeros = (vals == 0).mean()
    print(f"{str(layer)[:28]:28s} shape {str(tuple(h.shape)):10s} first values {np.round(vals[:4], 3)}"
          + (f"   ({zeros:.0%} are 0 after ReLU)" if isinstance(layer, nn.ReLU) else ""))

The network is not trained yet: its weights are random, so this output is noise. **Training only changes the numbers in the weight matrices**; the structure stays the same.

## 5. The training loop

The same loop as `forecast/train.py:51`, with the good habits of the repository:
- **mini-batches** from a `DataLoader`, shuffled each epoch;
- **AdamW** (Adam with weight decay) and a **OneCycle** learning-rate schedule;
- **gradient clipping**;
- **validation after every epoch**, and keep the **best epoch** (`warning/train.py:80`).


> 🧒 **In plain words:** training is practice with feedback, repeated: **(1)** take a handful of flash cards (a *batch*), **(2)** let the model guess, **(3)** measure how wrong it was (the *loss*), **(4)** work out which way to turn every knob (the *gradient*), **(5)** turn every knob a tiny bit (the *learning rate* sets how much). After all the cards (one *epoch*), check on cards it never practised on (*validation*). Keep the version that did best on those.

In [ ]:
def loader(split, batch=128, shuffle=False, n=None):
    x, y = X[split], Y[split]
    if n is not None:
        x, y = x[:n], y[:n]
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)), batch_size=batch, shuffle=shuffle)


def fit(model, epochs=60, lr=3e-3, weight_decay=1e-4, n_train=None, loss_fn=F.mse_loss, verbose=10):
    torch.manual_seed(0)
    tl, vl = loader("train", shuffle=True, n=n_train), loader("val")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tl))
    hist, best, best_state = [], np.inf, None
    t0 = time.time()
    for epoch in range(epochs):
        model.train()                                   # dropout on
        tr = 0.0
        for xb, yb in tl:                               # 1 a batch
            pred = model(xb)                            # 2 forward
            loss = loss_fn(pred, yb)                    # 3 loss
            opt.zero_grad()
            loss.backward()                             # 4 gradients by backpropagation
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()                    # 5 update the weights
            tr += loss.item() * len(xb)
        model.eval()                                    # dropout off
        with torch.no_grad():                           # 6 validation, no gradients
            va = sum(loss_fn(model(xb), yb).item() * len(xb) for xb, yb in vl) / len(vl.dataset)
        hist.append((tr / len(tl.dataset), va))
        if va < best:
            best, best_state, best_epoch = va, copy.deepcopy(model.state_dict()), epoch
        if verbose and (epoch % verbose == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:3d}  train {hist[-1][0]:.4f}  val {va:.4f}")
    model.load_state_dict(best_state)                   # keep the best epoch
    print(f"best epoch {best_epoch}, val loss {best:.4f}  ({time.time() - t0:.0f} s)")
    return np.array(hist), best_epoch


def predict(model, split):
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(X[split])).numpy()


def plot_curves(hist, best_epoch, title):
    plt.figure(figsize=(6, 3))
    plt.plot(hist[:, 0], label="train"); plt.plot(hist[:, 1], label="validation")
    plt.axvline(best_epoch, color="tab:green", ls="--", label="best epoch")
    plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss (MSE, scaled units)"); plt.title(title); plt.legend()

In [ ]:
net = mlp()
hist, be = fit(net)
plot_curves(hist, be, "MLP 288 → 64 → 64 → 8")
report("MLP 288-64-64-8", predict(net, "val"), count(net))

✏️ **Try:** change the learning rate to `3e-2` and to `3e-4`, and re-run the two cells above. What happens to the curves?

## 6. A model that uses the time order: a causal convolutional network

The MLP sees 288 numbers and does not know that neighbouring numbers are neighbouring moments. A **1-D convolution** slides the same small filter along time, so it needs far fewer parameters for a long input. **Causal** means the filter only looks back: padding is added on the left only, as in the repository's `CausalBlock` (`forecast/models.py:25`). **Dilations** 1, 2, 4, … double how far back each layer looks.


> 🧒 **In plain words:** a **convolution** is a small magnifying glass that slides along the time series and looks at a few neighbouring moments at a time. The same glass is used everywhere, so it needs few knobs. **Causal** means the glass only looks backwards in time, never at the future, just like real life.

In [ ]:
class CausalBlock(nn.Module):
    """Dilated causal convolution with a residual connection (simplified from forecast/models.py)."""

    def __init__(self, ch, dilation, kernel=3):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(ch, ch, kernel, dilation=dilation)

    def forward(self, x):                               # x: (batch, channels, time)
        return x + F.gelu(self.conv(F.pad(x, (self.pad, 0))))   # pad on the LEFT only: no future


class CausalCNN(nn.Module):
    def __init__(self, width=24, dilations=(1, 2, 4, 8, 16, 32)):
        super().__init__()
        self.stem = nn.Conv1d(3, width, 1)
        self.blocks = nn.Sequential(*[CausalBlock(width, d) for d in dilations])
        self.head = nn.Linear(width, HORIZON)

    def forward(self, x):                               # x: (batch, time, channels)
        h = self.blocks(self.stem(x.transpose(1, 2)))   # → (batch, width, time)
        return self.head(h[:, :, -1])                   # read out at the last time step = "now"


cnn = CausalCNN()
rf = 1 + 2 * sum((1, 2, 4, 8, 16, 32))
print(f"parameters {count(cnn):,}   receptive field {rf} steps = {rf / 4:.0f} h (input is {LOOKBACK} steps)")

# check, don't trust: changing the future must not change the output at "now"
x = torch.from_numpy(X["val"][:2]).clone()
y0 = cnn(x)
x_late = x.clone(); x_late[:, -1] += 5.0              # change only the newest sample
x_early = x.clone(); x_early[:, :10] += 5.0           # change only the oldest samples
print("output changes when the newest sample changes:", not torch.allclose(y0, cnn(x_late)))
print("output changes when the oldest samples change:", not torch.allclose(y0, cnn(x_early)))

Watch the shapes inside it:

In [ ]:
h = torch.from_numpy(X["val"][:1])
print(f"{'input':32s} {tuple(h.shape)}  (batch, time, channels)")
h = h.transpose(1, 2); print(f"{'transpose':32s} {tuple(h.shape)}  (batch, channels, time)")
h = cnn.stem(h); print(f"{'1×1 convolution: 3 → 24 channels':32s} {tuple(h.shape)}")
for b in cnn.blocks:
    h = b(h); print(f"{'causal block, dilation ' + str(b.conv.dilation[0]):32s} {tuple(h.shape)}")
h = h[:, :, -1]; print(f"{'take the last time step':32s} {tuple(h.shape)}")
h = cnn.head(h); print(f"{'linear head':32s} {tuple(h.shape)}  ← the 8 forecasts")

In [ ]:
hist, be = fit(cnn, epochs=40)
plot_curves(hist, be, "causal CNN")
report("causal CNN", predict(cnn, "val"), count(cnn))

## 7. Overfitting, on purpose

A big network (288 → 512 → 512 → 8, ≈ 415 000 parameters) trained on only **200** examples, without weight decay: far more knobs than examples.


> 🧒 **In plain words:** a student who memorises the answers of the practice test gets 100 % on it and fails the real exam. A model with far more knobs than examples does the same: it **memorises** instead of **understanding**. You see it when the training loss keeps falling while the validation loss goes up.

In [ ]:
big = mlp(hidden=(512, 512))
print(f"{count(big):,} parameters for 200 examples")
hist_big, be_big = fit(big, epochs=150, lr=1e-3, weight_decay=0.0, n_train=200, verbose=30)
plot_curves(hist_big, be_big, "too big, too little data: training loss falls, validation does not")

The training loss keeps falling: the network memorises the 200 examples. The validation loss stops improving early and then rises: what it memorised does not carry over. **Keeping the best epoch** (early stopping) saves the best model. Regularisation (dropout, strong weight decay) makes memorising harder:

In [ ]:
reg = mlp(hidden=(512, 512), dropout=0.5)
hist_reg, be_reg = fit(reg, epochs=150, lr=1e-3, weight_decay=0.3, n_train=200, verbose=30)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for a, h, title in ((ax[0], hist_big, "no regularisation"), (ax[1], hist_reg, "dropout 0.5 + weight decay 0.3")):
    a.plot(h[:, 0], label="train"); a.plot(h[:, 1], label="validation")
    a.set_yscale("log"); a.set_xlabel("epoch"); a.set_title(title); a.legend()
ax[0].set_ylabel("loss")
print(f"gap between validation and training loss at the end: {hist_big[-1, 1] / hist_big[-1, 0]:.0f}× without, "
      f"{hist_reg[-1, 1] / hist_reg[-1, 0]:.0f}× with regularisation")
report("big MLP, 200 ex., no reg.", predict(big, "val"), count(big))
report("big MLP, 200 ex., regularised", predict(reg, "val"), count(reg))

Regularisation closes much of the gap, but look at the validation loss itself: no model in this notebook gets far below ≈ 0.25 on it. **The limit here is the data, not the model:** the training set contains no storm, and the validation set is a storm. On a real plant, the cure would be more (and more varied) data, which is why the repository's forecaster is trained on 160 randomised influent scenarios (`forecast/dataset.py`, `build_corpus`).

## 8. The test set: once, at the end

Everything so far was chosen on the **validation** data (storm 1). The **test** data (storm 2) has not been looked at. Now, once:


> 🧒 **In plain words:** the test set is the final exam. You sit it **once**, after all your choices are made. If you look at it, change something and look again, it is no longer an exam.

In [ ]:
test_models = {"persistence": (persistence(X["test"]), 0), "same time yesterday": (yesterday(X["test"]), 0),
               "linear (ridge)": (lin.predict(flat(X["test"])), n_lin),
               "MLP 288-64-64-8": (predict(net, "test"), count(net)), "causal CNN": (predict(cnn, "test"), count(cnn))}
print(f"{'model':22s} {'params':>8s}  test MAE 15 min   2 h  [m³/d]")
for name, (p, n) in test_models.items():
    e = mae(p, "test")
    print(f"{name:22s} {n:>8,}  {e[0]:14.0f} {e[-1]:7.0f}")

In [ ]:
k = HORIZON - 1                                           # the 2-h-ahead forecast
t_target = T["test"] + k / 96
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(t_target, to_flow(Y["test"][:, k]), color="k", lw=2.2, label="what happened")
for name in ("persistence", "linear (ridge)", "causal CNN"):
    ax.plot(t_target, to_flow(test_models[name][0][:, k]), lw=1.2, label=f"{name}, 2 h ahead")
ax.set_xlabel("day"); ax.set_ylabel("Q [m³/d]"); ax.legend(ncol=2); ax.set_title("test: storm 2, forecasts made 2 hours earlier")

**Discuss:** When we ran this, the plain **linear model** had the lowest test error at 2 hours; check whether yours agrees. With about 1 900 examples, more flexible models had nothing more to learn and more to overfit. The project found the same thing at a larger scale: a carefully tuned PID matched its learned controller. Which model wins at 15 minutes, and which at 2 hours? What happens at the storm, which no model saw in training? What does this say about the training data you would need on a real plant?

### Save the model **with everything it needs**, then use it as a plant would

Weights alone are not enough: the scaler, the input definition and the settings travel with it, as in `forecast/train.py:85`. `forecast/predictor.py` then wraps the model for online use.

In [ ]:
torch.save({"state_dict": cnn.state_dict(), "mean": MEAN, "std": STD, "channels": ["Q", "S_NH", "S_S"],
            "lookback": LOOKBACK, "horizon": HORIZON, "model": "CausalCNN(width=24, dilations=1..32)"},
           DATA / "flow_forecaster.pt")

blob = torch.load(DATA / "flow_forecaster.pt", weights_only=False)
model = CausalCNN(); model.load_state_dict(blob["state_dict"]); model.eval()


def forecast(last_24h_raw):
    """last_24h_raw: (96, 3) array of Q [m³/d], S_NH, S_S [g/m³] as measured. Returns 8 flows [m³/d]."""
    z = (np.log(last_24h_raw) - blob["mean"]) / blob["std"]
    with torch.no_grad():
        out = model(torch.from_numpy(z[None].astype(np.float32)))[0].numpy()
    return np.exp(out * blob["std"][0] + blob["mean"][0])


raw_window = np.exp(X["test"][100] * STD + MEAN)          # pretend these came from the sensors
print("forecast for the next 2 h [m³/d]:", forecast(raw_window).round(0))
print("what really happened       [m³/d]:", to_flow(Y["test"][100]).round(0))

## 9. Extra: forecast a range instead of a number

A plant operator needs to know how sure the forecast is. The repository's forecaster outputs the 10 %, 50 % and 90 % **quantiles** and trains them with the **pinball loss** (`forecast/models.py:85`). Same model, one more output dimension, different loss:


> 🧒 **In plain words:** instead of "the flow will be 25 000", say "the flow will very likely be between 21 000 and 30 000". An operator can plan better with a range, the same way a weather forecast says "70 % chance of rain".

In [ ]:
QS = torch.tensor([0.1, 0.5, 0.9])

class QuantileCNN(CausalCNN):
    def __init__(self):
        super().__init__()
        self.head = nn.Linear(24, HORIZON * 3)

    def forward(self, x):
        return super().forward(x).view(-1, HORIZON, 3)     # (batch, horizon, quantile)


def pinball(pred, y):
    err = y.unsqueeze(-1) - pred
    return torch.maximum(QS * err, (QS - 1) * err).mean()


qcnn = QuantileCNN()
hist_q, be_q = fit(qcnn, epochs=40, loss_fn=pinball, verbose=20)
with torch.no_grad():
    q = qcnn(torch.from_numpy(X["test"])).numpy()
inside = ((Y["test"] >= q[..., 0]) & (Y["test"] <= q[..., 2])).mean()
print(f"share of test values inside the 10–90 % band: {inside:.0%} (ideal: 80 %)")
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.fill_between(t_target, to_flow(q[:, k, 0]), to_flow(q[:, k, 2]), alpha=0.3, label="10–90 % band, 2 h ahead")
ax.plot(t_target, to_flow(q[:, k, 1]), lw=1.2, label="median")
ax.plot(t_target, to_flow(Y["test"][:, k]), color="k", lw=2, label="what happened")
ax.legend(); ax.set_xlabel("day"); ax.set_ylabel("Q [m³/d]")

### ✅ What you just did

- read the input and output **shapes** and let them choose the model family and the loss
- beat (or failed to beat) **baselines** first
- counted **parameters** and compared them with the number of examples
- followed one example **layer by layer** through an MLP and a causal CNN
- wrote the **training loop**: batch → forward → loss → backward → update → validate → keep the best
- caused **overfitting** and fixed it
- used the **test set once**, saved the model **with its scaler**, reloaded it and forecast from raw measurements

**Next:** notebook 3, when there is no right answer to learn from (reinforcement learning).